# 03 Feature Engineering

**03 Notebook will house all our feature engineering efforts.**

+ Feature brainstorming
+ Sanity checks
+ Distribution checks
+ General dataset behavioral characteristic/trend analysis

In [4]:
%load_ext autoreload
%autoreload 2

import sys 
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

sys.path.insert(0, '../src')

from icu_tft.data.connect import get_connection
from icu_tft.data.static_features import (
    build_static_features,
    feature_summary,
    FEATURE_GROUPS,
)

from icu_tft.data.extract_timeseries import (
    build_timeseries,
    validate_cohort,
    ALL_FEATURES,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(acstime)s %(name)s %(levelname)s %(message)s',
    datefmt='%H:%M:%S',
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':130, 'savefig.dpi':300})

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# cds

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('reports/figures')
COHORT_PQ = PROCESSED_DIR / 'cohort.parquet'
STATIC_PQ = PROCESSED_DIR / 'static_features.parquet'
TS_PQ = PROCESSED_DIR / 'timeseries_features.parquet'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('paths ok:')
print(f'cohort  : {COHORT_PQ}')
print(f'static : {STATIC_PQ} ')
print(f'ts : {TS_PQ}')


paths ok:
cohort  : ../data/processed/cohort.parquet
static : ../data/processed/static_features.parquet 
ts : ../data/processed/timeseries_features.parquet
final : ../data/processed/final_static_df.parquet 


# Cohort Loading and registering DuckDB views

In [6]:
con = get_connection()

assert COHORT_PQ.exists(), (
    f'cohort.parquet not found at {COHORT_PQ}.\n'
    f'Run notebook 91 that extracts cohort first. '
)

[connect.py] Registered 30 views against mimic.duckdb
[connect.py] WARNING — 1 file(s) not found (views skipped):
  /Users/longer/ICU_Mortality_Prediction/data/raw/hosp/antimicrobial.csv.gz


In [7]:
cohort = pd.read_parquet(COHORT_PQ)

print(f' cohort loaded. {cohort.shape[0]:,} stays x {cohort.shape[1]} columns')
print(f' columns: {list(cohort.columns)}')
print('===' * 30)
print('Target Label Distribution (mortality_24h)')
vc = cohort['mortality_24h'].value_counts()
for label, count in vc.items():
    print(f'    {label}: {count:,} ({count / len(cohort):.1%})')

 cohort loaded. 67,223 stays x 14 columns
 columns: ['subject_id', 'hadm_id', 'stay_id', 'gender', 'anchor_age', 'admission_type', 'first_careunit', 'icu_los_hours', 'hospital_los_hours', 'mortality_24h', 'mortality_inhospital', 'insurance', 'race', 'marital_status']
Target Label Distribution (mortality_24h)
    0: 66,472 (98.9%)
    1: 751 (1.1%)


# Static Features

In [8]:
import icu_tft.data.static_features as _sf
import types

# types package needed for fixing up one of the import functions

def _extract_severity_proxies_updated(cohort: pd.DataFrame, con) -> pd.DataFrame:
    '''updated and optimized version of the severity proxy extraction function.'''
    stay_ids = tuple(cohort['stay_id'].unique().tolist())
    
    query = f'''
        SELECT 
            ce.stay_id, 
            MIN(CASE WHEN ce.itemid = 220052 THEN ce.valuenum END) AS severity_min_map_6h,
            MAX(CASE WHEN ce.itemid = 50813  THEN ce.valuenum END) AS severity_max_lactate_6h
        FROM mimic_icu.chartevents ce
        INNER JOIN mimic_icu.icustays ie ON ce.stay_id = ie.stay_id
        WHERE ce.stay_id IN {stay_ids}
            AND ce.itemid IN (220052, 50813)
            AND ce.charttime >= ie.intime
            AND ce.charttime <= ie.intime + INTERVAL '6 hours'
            AND ce.valuenum IS NOT NULL
        GROUP BY ce.stay_id
    '''
    proxies_df = con.execute(query).df()
    
    gcs_query = f'''
        WITH gcs_components AS (
            SELECT 
                ce.stay_id, 
                ce.charttime,
                SUM(ce.valuenum) AS gcs_total
            FROM mimic_icu.chartevents ce
            INNER JOIN mimic_icu.icustays ie ON ce.stay_id = ie.stay_id
            WHERE ce.stay_id IN {stay_ids}
                AND ce.itemid IN (223900, 223901, 220739)
                AND ce.charttime >= ie.intime 
                AND ce.charttime <= ie.intime + INTERVAL '6 hours'
            GROUP BY ce.stay_id, ce.charttime
            HAVING COUNT(DISTINCT ce.itemid) = 3
        )
        SELECT 
            stay_id, 
            MIN(gcs_total) AS severity_min_gcs_6h
        FROM gcs_components
        GROUP BY stay_id
    '''
    gcs_df = con.execute(gcs_query).df()


    merged = pd.merge(cohort[['stay_id']], proxies_df, on='stay_id', how='left')
    merged = pd.merge(merged, gcs_df, on='stay_id', how='left')
    return merged

_sf.extract_severity_proxies = _extract_severity_proxies_updated
print('Severity proxy function patched (gcs_query fix applied).')
    

Severity proxy function patched (gcs_query fix applied).


In [12]:
ts_df = pd.read_parquet(TS_PQ)

ts_features = [col for col in ts_df.columns if col not in ['stay_id', 'time_step']]

print(f'Applying LOCF imputation across {len(ts_features)} timeseries features.')

ts_df = ts_df.sort_values(by=['stay_id', 'time_step'])

ts_df[ts_features] = ts_df.groupby('stay_id')[ts_features].ffill()
print('Aggregating 24-hour means')

ts_means_df = ts_df.groupby('stay_id')[ts_features].mean().reset_index()

new_col_names = {col: f'{col}_mean24h' for col in ts_features}
ts_means_df = ts_means_df.rename(columns=new_col_names)

print(f'Extracted flattened timeseries for {len(ts_means_df)}')
display(ts_means_df.head())


Applying LOCF imputation across 36 timeseries features.
Aggregating 24-hour means
Extracted flattened timeseries for 67223


,stay_id,heart_rate_mean24h,sbp_mean24h,dbp_mean24h,mbp_mean24h,spo2_mean24h,resp_rate_mean24h,temperature_c_mean24h,gcs_total_mean24h,creatinine_mean24h,...,creatinine_missing_mean24h,bun_missing_mean24h,sodium_missing_mean24h,potassium_missing_mean24h,bicarbonate_missing_mean24h,lactate_missing_mean24h,wbc_missing_mean24h,hemoglobin_missing_mean24h,platelets_missing_mean24h,inr_missing_mean24h
0,30000153,107.500000,120.916664,NaN,87.565216,96.625000,15.125000,37.543480,12.500000,NaN,...,1.000000,0.416667,1.0,1.0,0.708333,0.708333,0.416667,0.416667,1.0,1.000000
1,30000213,82.000000,133.791672,NaN,NaN,99.041664,19.083334,37.046299,13.416667,3.6875,...,0.333333,0.333333,1.0,1.0,1.000000,1.000000,1.000000,0.916667,1.0,1.000000
2,30000484,89.565216,105.086960,NaN,NaN,99.782608,14.652174,35.884056,7.086957,NaN,...,1.000000,0.708333,1.0,1.0,1.000000,1.000000,0.708333,0.708333,1.0,0.708333
3,30000646,83.500000,93.083336,NaN,NaN,95.916664,22.875000,37.232807,15.000000,NaN,...,1.000000,1.000000,1.0,1.0,0.708333,1.000000,1.000000,0.250000,1.0,0.541667
4,30000831,92.208336,109.416664,NaN,NaN,94.958336,27.041666,37.361111,10.916667,2.2000,...,0.125000,0.125000,1.0,1.0,1.000000,1.000000,0.416667,0.416667,1.0,0.416667


In [27]:
static_df = pd.read_parquet(STATIC_PQ)


## SOFA Score Extraction

**Objs**
+ Sequential Organ Failure Assessment (SOFA)
    + Most important factor going into predicting ICU mortality
    + Takes together the severity of dysfunction of each 6 organ systems




In [31]:
print("Calculating 24h Proxy SOFA score from available features...")

def calculate_proxy_sofa(row):
    score = 0
    
    if 'spo2_mean24h' in row and pd.notnull(row['spo2_mean24h']) and 'fio2_mean24h' in row and pd.notnull(row['fio2_mean24h']):
        ratio = row['spo2_mean24h'] / (row['fio2_mean24h'] / 100)
        if ratio < 150: score += 4
        elif ratio < 235: score += 3
        elif ratio < 315: score += 2
        elif ratio < 400: score += 1
            
    if 'platelets_mean24h' in row and pd.notnull(row['platelets_mean24h']):
        plt = row['platelets_mean24h']
        if plt < 20: score += 4
        elif plt < 50: score += 3
        elif plt < 100: score += 2
        elif plt < 150: score += 1

    if 'bilirubin_mean24h' in row and pd.notnull(row['bilirubin_mean24h']):
        bili = row['bilirubin_mean24h']
        if bili >= 12.0: score += 4
        elif bili >= 6.0: score += 3
        elif bili >= 2.0: score += 2
        elif bili >= 1.2: score += 1

    if 'mbp_mean24h' in row and pd.notnull(row['mbp_mean24h']):
        map_val = row['mbp_mean24h']
        if map_val < 70: score += 1

    if 'gcs_total_mean24h' in row and pd.notnull(row['gcs_total_mean24h']):
        gcs = row['gcs_total_mean24h']
        if gcs < 6: score += 4
        elif gcs <= 9: score += 3
        elif gcs <= 12: score += 2
        elif gcs <= 14: score += 1

    if 'creatinine_mean24h' in row and pd.notnull(row['creatinine_mean24h']):
        creat = row['creatinine_mean24h']
        if creat >= 5.0: score += 4
        elif creat >= 3.5: score += 3
        elif creat >= 2.0: score += 2
        elif creat >= 1.2: score += 1

    return score

sofa_df = ts_means_df[['stay_id']].copy()
sofa_df['sofa_24h'] = ts_means_df.apply(calculate_proxy_sofa, axis=1)

print(f"Calculated SOFA for {len(sofa_df):,} patients.")
print(f"SOFA Score Distribution (Proxy):\n{sofa_df['sofa_24h'].describe()}")


Calculating 24h Proxy SOFA score from available features...
Calculated SOFA for 67,223 patients.
SOFA Score Distribution (Proxy):
count    67223.000000
mean         2.331732
std          2.041805
min          0.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         12.000000
Name: sofa_24h, dtype: float64


## Join and Merge Final Dataset

**Obj**
+ Merge the cleaned static features, flattened time series means, and SOFA score benchmarks into one singular dataset
+ Export this final matrix of data to 'train_data.parquet'
    + Will be called in future modeling efforts

In [29]:
final_static_df = pd.read_parquet(STATIC_PQ)

final_static_df = pd.merge(
    final_static_df, 
    cohort[['stay_id', 'mortality_24h', 'mortality_inhospital']], 
    on='stay_id', 
    how='inner'
)

print(f'static features shape: {final_static_df.shape}')
print(f'time series means shape: {ts_means_df.shape}')
print(f'sofa score features shape: {sofa_df.shape}')

ml_data = pd.merge(final_static_df, ts_means_df, on='stay_id', how='inner')
ml_data = pd.merge(ml_data, sofa_df, on='stay_id', how='inner')

if 'mortality_24h' not in ml_data.columns:
    print("WARNING: Target label 'mortality_24h' missing from final dataset.")
else:
    print("Success: Target label 'mortality_24h' is present and ready for modeling.")

print(f'Final Dataset shape: {ml_data.shape[0]:,} rowx x {ml_data.shape[1]:,} columns.')

static features shape: (67223, 54)
time series means shape: (67223, 37)
sofa score features shape: (67223, 2)
Success: Target label 'mortality_24h' is present and ready for modeling.
Final Dataset shape: 67,223 rowx x 91 columns.


In [30]:
TRAIN_DATA_PQ = PROCESSED_DIR / 'train_data.parquet'
ml_data.to_parquet(TRAIN_DATA_PQ, index=False)
print(f'Exported ready dataset to {TRAIN_DATA_PQ}.')


Exported ready dataset to ../data/processed/train_data.parquet.
